In [ ]:
import os, sys

parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.colors as mcolors
import re
import scienceplots
plt.style.use(["science", "grid"])
title_size = 30
axis_size = 26
tick_size = 24
linewidth = 3

from utils import create_df
save = False

In [ ]:
def blend_with_white(color, t):
    """t in [0,1]; 0=color, 1=white"""
    rgb = np.array(mcolors.to_rgb(color))
    return tuple((1 - t) * rgb + t * np.ones(3))

def is_gcaci(method: str) -> bool:
    return "gcaci" in method

def parse_lr(method: str) -> float:
    m = re.search(r"lr=([0-9]*\.?[0-9]+)", method)
    return float(m.group(1))

In [ ]:
import math
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


# ============================================================
# Colors
# ============================================================

# POGO color
OTHER_COLOR = "#0000CC"

# GCACI colors: distinct, no blue, no gray/black
GCACI_COLORS = {
    0.01: "#E69F00",  # orange
    0.05: "#D55E00",  # vermillion
    0.10: "#009E73",  # green
    0.25: "#CC79A7",  # reddish purple
    0.50: "#8C564B",  # brown
    0.75: "#E7298A",  # magenta
    1.00: "#BCBD22",  # olive
}


# ============================================================
# Helper functions
# ============================================================

def is_gcaci(method: str) -> bool:
    return method.startswith("gc_aci_lr=")


def parse_lr(method: str) -> float:
    return float(method.split("gc_aci_lr=")[1])


def color_for_method(method: str):
    if is_gcaci(method):
        lr = parse_lr(method)
        return GCACI_COLORS[lr]
    return OTHER_COLOR


def order_for_rowwise_display(items, ncol):
    """
    Matplotlib legends fill entries column-wise.
    This reorders a desired row-wise list so it appears row-wise.
    """
    N = len(items)
    nrow = math.ceil(N / ncol)

    pad = nrow * ncol - N
    items = list(items) + [None] * pad

    grid = [
        items[r * ncol:(r + 1) * ncol]
        for r in range(nrow)
    ]

    out = []
    for c in range(ncol):
        for r in range(nrow):
            v = grid[r][c]
            if v is not None:
                out.append(v)

    return out


# ============================================================
# Main legend generation function
# ============================================================

def make_legend_only(savepath="legend_only.pdf"):
    linewidth = 2.5

    # --------------------------------------------------------
    # Method order
    # --------------------------------------------------------

    methods = [
        "cw_group_up",
        "gc_aci_lr=0.01",
        "gc_aci_lr=0.05",
        "gc_aci_lr=0.1",
        "gc_aci_lr=0.25",
        "gc_aci_lr=0.5",
        "gc_aci_lr=0.75",
        "gc_aci_lr=1",
    ]

    # --------------------------------------------------------
    # Pretty labels
    # --------------------------------------------------------

    pretty = {
        "cw_group_up": r"POGO",
        "gc_aci_lr=0.01": r"GCACI ($\eta$ = 0.01)",
        "gc_aci_lr=0.05": r"GCACI ($\eta$ = 0.05)",
        "gc_aci_lr=0.1":  r"GCACI ($\eta$ = 0.10)",
        "gc_aci_lr=0.25": r"GCACI ($\eta$ = 0.25)",
        "gc_aci_lr=0.5":  r"GCACI ($\eta$ = 0.50)",
        "gc_aci_lr=0.75": r"GCACI ($\eta$ = 0.75)",
        "gc_aci_lr=1":    r"GCACI ($\eta$ = 1.0)",
    }

    # --------------------------------------------------------
    # Reference handles: first column
    # --------------------------------------------------------

    ref_handles = [
        Line2D(
            [0], [0],
            color="k",
            linestyle="--",
            linewidth=4,
        ),
        Line2D(
            [0], [0],
            color="0.55",
            linestyle=":",
            linewidth=4,
        ),
    ]

    ref_labels = [
        r"Ideal",
        r"Ideal $\pm3\%$",
    ]

    # --------------------------------------------------------
    # Method handles
    # --------------------------------------------------------

    method_handles = [
        Line2D(
            [0], [0],
            color=color_for_method(m),
            marker="o",
            linestyle="-",
            linewidth=linewidth + 2,
            markersize=10,
        )
        for m in methods
    ]

    method_labels = [
        pretty[m]
        for m in methods
    ]

    # --------------------------------------------------------
    # Build desired two-row visual layout
    # --------------------------------------------------------

    nrows = 2
    ncol_total = 1 + math.ceil(len(method_handles) / nrows)

    # Desired visual layout:
    #
    # Row 1:
    # Ideal        POGO          GCACI 0.05    GCACI 0.25    GCACI 0.75
    #
    # Row 2:
    # Ideal ±3%    GCACI 0.01    GCACI 0.10    GCACI 0.50    GCACI 1.0

    row1_handles = [ref_handles[0]] + method_handles[0::2]
    row2_handles = [ref_handles[1]] + method_handles[1::2]

    row1_labels = [ref_labels[0]] + method_labels[0::2]
    row2_labels = [ref_labels[1]] + method_labels[1::2]

    handles_rowwise = row1_handles + row2_handles
    labels_rowwise = row1_labels + row2_labels

    # Matplotlib fills column-wise, so reorder to force row-wise display.
    handles = order_for_rowwise_display(handles_rowwise, ncol_total)
    labels = order_for_rowwise_display(labels_rowwise, ncol_total)

    # --------------------------------------------------------
    # Create legend-only figure
    # --------------------------------------------------------

    plt.rcParams.update({
        "font.family": "serif",
        "mathtext.fontset": "cm",
    })

    fig, ax = plt.subplots(figsize=(15.5, 1.55))
    ax.axis("off")

    leg = ax.legend(
        handles,
        labels,
        loc="center",
        frameon=False,
        ncol=ncol_total,
        handlelength=2.1,
        handletextpad=0.7,
        columnspacing=1.7,
        fontsize=20,
    )

    fig.tight_layout()

    if savepath is not None:
        fig.savefig(savepath, bbox_inches="tight")

    return fig, ax, leg


# ============================================================
# Generate and save legend
# ============================================================

fig, ax, leg = make_legend_only("legend_only.pdf")

# Optional: also save as PNG
fig.savefig("legend_only.pdf", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
# Load results
dgp_name = "k=100/bounded_no_changepoint"
df = create_df(dgp_name)

# Isolate methods to plot
df = df[df["method"] != "up-ocp"]

# # Plot
# if dgp_name == "bounded_changepoint":
#     y_r = 50
# else:
#     y_r = 10

In [ ]:
# Plot
methods = sorted(df["method"].unique())
gcaci_methods = [m for m in methods if is_gcaci(m)]
other_methods = [m for m in methods if not is_gcaci(m)]

# ---- GC-ACI: RED (single hue) with intensity by LR ----
gcaci_cmap = mpl.cm.Reds
lrs = np.array([parse_lr(m) for m in gcaci_methods], dtype=float)
norm = mpl.colors.Normalize(vmin=float(lrs.min()), vmax=float(lrs.max()))
def gcaci_color(method: str):
    u = norm(parse_lr(method))   # 0..1
    u = 0.30 + 0.60 * u          # avoid too-white/too-dark extremes
    return gcaci_cmap(u)


# ---- Other methods: GREEN (distinct shades per method) ----
# other_cmap = mpl.cm.summer
# # pick evenly spaced greens, one per other method
# greens = [other_cmap(u) for u in np.linspace(0.45, 0.85, max(1, len(other_methods)))]
# other_color = {m: greens[i] for i, m in enumerate(other_methods)}

COLOR=  "#0700C8"   # bright green
other_color = {m: COLOR for m in other_methods} 

def color_for_method(method: str):
    return gcaci_color(method) if is_gcaci(method) else other_color[method]

def label_for_method(method: str):
    return f"GC-ACI lr={parse_lr(method):g}" if is_gcaci(method) else method

# ---------- Common plot settings ----------
percent_fmt = mtick.PercentFormatter(xmax=1.0, decimals=0)

# -------- Figure 1 --------
fig1, ax1 = plt.subplots(figsize=(8.5, 6.5))
COV_LO, COV_HI = 0.75, 0.95

for method, g in df.groupby("method"):
    g = g.copy()

    # keep ONLY points inside the band
    in_band = (g["avg_worst_group_coverage"] >= COV_LO) & (g["avg_worst_group_coverage"] <= COV_HI)
    g = g[in_band].copy()
    if g.empty:
        continue

    ax1.errorbar(
        g["avg_set_size_mean"].to_numpy(),
        g["avg_worst_group_coverage"].to_numpy(),
        xerr=g["avg_set_size_mean_sem"].to_numpy(),
        yerr=g["avg_worst_group_coverage_sem"].to_numpy(),
        fmt="o-",
        color=color_for_method(method),
        linewidth=linewidth,
        markersize=5,
        capsize=3,
        elinewidth=1,
        label=label_for_method(method),
    )
ax1.yaxis.set_major_formatter(percent_fmt)
ax1.set_xscale("log")
ax1.xaxis.set_minor_locator(mtick.LogLocator(base=10, subs=(5.0,)))
ax1.xaxis.set_minor_formatter(mtick.NullFormatter())


ax1.set_ylim([COV_LO-0.01, COV_HI+0.01])
ax1.set_title("Group Coverage vs. Radius Size", fontsize=title_size)
ax1.set_xlabel("Average Radius Size", fontsize=axis_size)
ax1.set_ylabel("Lowest Group Coverage Rate", fontsize=axis_size)
ax1.tick_params(axis="x", which='both', labelsize=tick_size, pad=10)
ax1.tick_params(axis="y", which='major', labelsize=tick_size, pad=10)
ax1.grid(True, linestyle="-", linewidth=1, alpha=0.5)
fig1.tight_layout()

if save == True:
    fig1.savefig(os.path.join("figures", f"{dgp_name}_cov_vs_rad.pdf"), dpi=300, bbox_inches="tight")

In [ ]:
# -------- Figure 2 --------
fig2, ax2 = plt.subplots(figsize=(8.5, 6.5))
for method, g in df.groupby("method"):
    g = g.copy()

    # keep ONLY points inside the band
    in_band = (g["avg_worst_group_coverage"] >= COV_LO) & (g["avg_worst_group_coverage"] <= COV_HI)
    g = g[in_band].copy()
    if g.empty:
        continue

    ax2.errorbar(
        g["avg_longest_group_error_seq"].to_numpy(),
        g["avg_worst_group_coverage"].to_numpy(),
        xerr=g["avg_longest_group_error_seq_sem"].to_numpy(),
        yerr=g["avg_worst_group_coverage_sem"].to_numpy(),
        fmt="o-",
        color=color_for_method(method),
        linewidth=linewidth,
        markersize=5,
        capsize=3,
        elinewidth=1,
        label=label_for_method(method),
    )

ax2.yaxis.set_major_formatter(percent_fmt)
ax2.set_xscale("log")

ax2.set_ylim([COV_LO-0.01, COV_HI+0.01])
ax2.set_title("Group Coverage vs. Adaptivity", fontsize=title_size)
ax2.set_xlabel("Max Consecutive Miscovers (Across All Groups)", fontsize=axis_size)
ax2.set_ylabel("Lowest Group Coverage Rate", fontsize=axis_size)
ax2.tick_params(axis="x", which='both', labelsize=tick_size, pad=10)
ax2.tick_params(axis="y", which='major', labelsize=tick_size, pad=10)
ax2.grid(True, linestyle="-", linewidth=1, alpha=0.5)

fig2.tight_layout()
if save == True:
    fig2.savefig(os.path.join("figures", f"{dgp_name}_cov_vs_adapt.pdf"), dpi=300, bbox_inches="tight")

In [ ]:
# -------- Figure 3 --------
fig3, ax3 = plt.subplots(figsize=(8.5, 6.5))
alpha_grid = np.linspace(0.01, 0.25, 49)
desired_cov_grid = np.sort(1.0 - alpha_grid)        # 0.75 ... 0.99

for method, g in df.groupby("method"):
    y_raw = g["avg_worst_group_coverage"].to_numpy()
    se_raw = g["avg_worst_group_coverage_sem"].to_numpy()

    # order = np.argsort(y_raw)     # sort by observed coverage
    y_all = y_raw
    se_all = se_raw

    m = min(len(y_all), len(desired_cov_grid))
    x  = desired_cov_grid
    y  = y_all[:m]
    se = se_all[:m]

    color = color_for_method(method)
    ax3.plot(
        x, y,
        "-o",
        color=color,
        linewidth=linewidth,
        markersize=5,
        label=label_for_method(method),
    )

# reference: perfect calibration
grid = np.linspace(0.75, 0.99, 400)
ax3.plot(grid, grid, "k--", linewidth=4, label="Observed = Desired")

# ±0.05 tolerance (duller grey, dotted)
tol = 0.03
grey = "0.55"  # dull grey
ax3.plot(grid, np.clip(grid - tol, 0, 1), ":", color=grey, linewidth=4, label="±0.05 tolerance")
ax3.plot(grid, np.clip(grid + tol, 0, 1), ":", color=grey, linewidth=4)

ax3.set_xlim([0.75-1e-3, 0.97])
ax3.set_ylim([0.75-1e-3, 0.97])
ax3.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0, decimals=0))
ax3.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0, decimals=0))
ax3.tick_params(axis="both", which="major", labelsize=tick_size, pad=10)
ax3.grid(True, linestyle="-", linewidth=1, alpha=0.5)

ax3.set_xlabel("Desired Coverage", fontsize=axis_size)
ax3.set_ylabel("Lowest Group Coverage", fontsize=axis_size)
ax3.set_title("Observed vs. Desired Coverage", fontsize=title_size)

if save == True:
    fig3.tight_layout()
    fig3.savefig(os.path.join("figures", f"{dgp_name}_cov_vs_dcov.pdf"), dpi=300, bbox_inches="tight")